# <span style="color:red; font-size:30px"> Lecture 18b: Merging </span>

<font size = "4">

- Datasets are often spread across mutiple files.

- Multi-file datasets can be visualized with an "Entity Relationship Diagram" (ERD). 

- An ERD depicts how the identifiers in each table are connected.

- Here is an example ERD for 3 files in the Formula One dataset

<img src="f1_data/erd_f1_simple.png" alt="drawing" width="600"/>

- Let's examine this relationship in more detail

In [1]:
import pandas as pd

df_results = pd.read_csv("f1_data/results.csv")
df_races = pd.read_csv("f1_data/races.csv")
df_circuits = pd.read_csv("f1_data/circuits.csv")

In [ ]:
print("df_results:")
display(df_results)
print("df_races:")
display(df_races)
print("df_circuits:")
display(df_circuits)

<font size = "4">

- We can use these 3 separate data frames together to gain information about a race result.

In [ ]:
# pick one of the rows in results.csv between 0 and 25,659

result_row = 25
# result_row = 3000
# result_row = 24001
race_result = df_results.iloc[result_row, :]

print("race_result:", result_row)

# grab the raceId number
race_id = race_result["raceId"]
print("race ID:", race_id)

# use the raceId number to grab the corresponding row of the races.csv dataset
race_info = df_races.query("raceId == @race_id")

race_year = race_info["year"].item()
race_name = race_info["name"].item()
circuit_id = race_info["circuitId"].item()

print("Year:", race_year)
print("Name of Race:", race_name)

# use circuitId number to grab the corresponding row of the circuits.csv dataset
circuit_info = df_circuits.query("circuitId == @circuit_id")
circuit_name = circuit_info["name"].item()
circuit_loc = circuit_info["location"].item()
circuit_country = circuit_info["country"].item()

print("Circuit Name:", circuit_name)
print("Location:", circuit_loc)
print("Country:", circuit_country)

<font size = "4">

- Navigating between 3 different DataFrames is not ideal. 

- It would be nice if we could combine the DataFrames in some way.

- We can't "stack" them together (called concatenation) because the first DataFrame has 25,660 rows, the second has 1,079 rows, and the third only has 76 rows.

- However, we can "merge" them together using the Pandas function `pd.merge`

- We'll begin by working with some smaller DataFrames just for visualization's sake

In [3]:
results_short = df_results[["resultId", "raceId", "driverId", "points"]]
races_short = df_races[["raceId", "year", "circuitId", "name"]]
circuits_short = df_circuits[["circuitId", "name", "location", "country"]]

print("results_short:")
display(results_short)
print("races_short:")
display(races_short)
print("circuits_short:")
display(circuits_short)

results_short:


,resultId,raceId,driverId,points
0,1,18,1,10.0
1,2,18,2,8.0
2,3,18,3,6.0
3,4,18,4,5.0
4,5,18,5,4.0
...,...,...,...,...
25655,25661,1086,825,0.0
25656,25662,1086,848,0.0
25657,25663,1086,849,0.0
25658,25664,1086,852,0.0


races_short:


,raceId,year,circuitId,name
0,1,2009,1,Australian Grand Prix
1,2,2009,2,Malaysian Grand Prix
2,3,2009,17,Chinese Grand Prix
3,4,2009,3,Bahrain Grand Prix
4,5,2009,4,Spanish Grand Prix
...,...,...,...,...
1074,1092,2022,22,Japanese Grand Prix
1075,1093,2022,69,United States Grand Prix
1076,1094,2022,32,Mexico City Grand Prix
1077,1095,2022,18,Brazilian Grand Prix


circuits_short:


,circuitId,name,location,country
0,1,Albert Park Grand Prix Circuit,Melbourne,Australia
1,2,Sepang International Circuit,Kuala Lumpur,Malaysia
2,3,Bahrain International Circuit,Sakhir,Bahrain
3,4,Circuit de Barcelona-Catalunya,Montmeló,Spain
4,5,Istanbul Park,Istanbul,Turkey
...,...,...,...,...
71,75,Autódromo Internacional do Algarve,Portimão,Portugal
72,76,Autodromo Internazionale del Mugello,Mugello,Italy
73,77,Jeddah Corniche Circuit,Jeddah,Saudi Arabia
74,78,Losail International Circuit,Al Daayen,Qatar


<font size = "4">

- We will combine `results_short` and `races_short` in the following way.

- Here are rows 0, 23, and 46 of `results_short` (you can get them using .iloc)


<div align="center">

|  | resultId | raceId | driverId | points |
| -------- | -------- | -------- | -------- | -------- |
| 0   | 1  | 18 | 1 | 10 |
| 23   | 24  | 19 | 9 | 8 |
| 46   | 47  | 20 | 9 | 6 |

</div>

- Here are rows 17, 18, and 19 of `races_short` (again, use .iloc)

<div align="center">

|  | raceId | year | circuitId | name |
| -------- | -------- | -------- | -------- | -------- |
| 17   | 18  | 2008 | 1 | Australian Grand Prix |
| 18   | 19  | 2008 | 2 | Malaysian Grand Prix |
| 19   | 20  | 2008 | 3 | Bahrain Grand Prix |

</div>

- After we use `pd.merge` to create a new DataFrame called `df_merged`, rows 0, 23, and 46 of the new DataFrame will be:

<div align="center">

|  | resultId | raceId | driverId | points | year | circuitId | name |
| -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- |
| 0   | 1  | 18 | 1 | 10 | 2008 | 1 | Australian Grand Prix | 
| 23   | 24  | 19 | 9 | 8 | 2008 | 2 | Malaysian Grand Prix |
| 46   | 47  | 20 | 9 | 6 | 2008 | 3 | Bahrain Grand Prix |

</div>


In [4]:
df_merged = pd.merge(left = results_short, right = races_short, how = "left", on = "raceId")

display(df_merged)
print()
display(df_merged.iloc[[0, 23, 46], :])

,resultId,raceId,driverId,points,year,circuitId,name
0,1,18,1,10.0,2008,1,Australian Grand Prix
1,2,18,2,8.0,2008,1,Australian Grand Prix
2,3,18,3,6.0,2008,1,Australian Grand Prix
3,4,18,4,5.0,2008,1,Australian Grand Prix
4,5,18,5,4.0,2008,1,Australian Grand Prix
...,...,...,...,...,...,...,...
25655,25661,1086,825,0.0,2022,11,Hungarian Grand Prix
25656,25662,1086,848,0.0,2022,11,Hungarian Grand Prix
25657,25663,1086,849,0.0,2022,11,Hungarian Grand Prix
25658,25664,1086,852,0.0,2022,11,Hungarian Grand Prix


,resultId,raceId,driverId,points,year,circuitId,name
0,1,18,1,10.0,2008,1,Australian Grand Prix
23,24,19,9,8.0,2008,2,Malaysian Grand Prix
46,47,20,9,6.0,2008,3,Bahrain Grand Prix


<font size = "4">

- We used 4 keyword arguments in `pd.merge`

- "left" and "right" are the two DataFrames we are merging. The choice of which one is "left" and which one is "right" will affect the ultimate order of the columns in the merged DataFrame.

- "how" is a string that describes how the merge is performed. For this class, we will just focus on "left" and "right". (Other advanced choices include "outer", "inner", "cross", "left_anti", "right_anti")

- When we do `how = "left"`, we are basically treating the "left" DataFrame as the primary one. The order of the rows is chosen to maintain the order of the data in the left DataFrame.

- "on" is a string corresponding to the merging variable. In this case we are merging based on "raceId". Rows in the two datasets with identical values of raceId are matched, and the additional info from the "right" DataFrame is added on to the "left" DataFrame.

<font size = "4">

**Exercise:** Run the following cell and print/display the outputs to see how the merging procedure works for other choices of "left", "right", and "how".

In [5]:
# order changes
df_merged_2 = pd.merge(left = results_short, right = races_short, how = "right", on = "raceId")

df_merged_3 = pd.merge(left = races_short, right = results_short, how = "left", on = "raceId")

df_merged_4 = pd.merge(left = races_short, right = results_short, how = "right", on = "raceId")

<font size = "4">

- Next, we perform an additional merge so that we include the data from `circuits_short`

In [ ]:
df_merged_final = pd.merge(left = df_merged, right = circuits_short, how = "left", on = "circuitId")

display(df_merged_final.head())


<font size = "4">

<font size = "4">

Do you notice anything odd about the column names?

<font size = "4">

Instead of creating the additional variables `results_short`, `races_short`, and `circuits_short` we could have directly subsetted the desired columns when passing in the input arguments to `pd.merge`. 